<a href="https://colab.research.google.com/github/mdshahabuddin-ml/FreightQuote_AI/blob/Milestone2/Copy_of_Infosys_Milestone1_Authentication_Maritime_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Infosys Springboard Internship 7.0 — Milestone 1
## User Authentication Module — Maritime AI Glass UI

This notebook launches a Streamlit app with Login, Sign Up, Forgot Password (security question and Gmail OTP), a JWT-protected User Dashboard, and a separate Admin Dashboard.

**Before running:** create the six secrets listed below in the Colab Secrets panel (key icon), and enable notebook access for each. Do not paste secret values into notebook cells.

- `JWT_SECRET` — long random secret, at least 32 characters
- `NGROK_AUTHTOKEN` — token from the ngrok dashboard
- `EMAIL_ADDRESS` — Gmail address used to send OTPs
- `EMAIL_PASSWORD` — Gmail App Password (requires Google 2-Step Verification)
- `ADMIN_USERNAME` — separate admin username
- `ADMIN_PASSWORD` — strong password for the admin login

The assignment says to keep admin credentials separate from signup accounts but also says to remove hard-coded admin passwords before submission. This notebook uses Colab Secrets for both admin values so the password is not embedded in the notebook.

**Important:** This is an educational demo. The JSON user database is temporary and lives only for the active Colab runtime. Use test accounts only. Do not expose secrets or real personal information. The app's public URL is accessible to anyone who has it while the ngrok tunnel is running.

In [ ]:
%pip -q install streamlit PyJWT pyngrok

## Create the Streamlit application

Run the next cell once. It writes the app source to `/content/app.py`. The source intentionally reads all credentials from environment variables populated from Colab Secrets in the launch cell.

In [ ]:
%%writefile /content/app.py
import hashlib
import hmac
import json
import os
import re
import secrets
import smtplib
import time
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage
from pathlib import Path
from threading import RLock

import jwt
import streamlit as st

# Configuration is supplied by the notebook through Colab Secrets / environment variables.
JWT_SECRET = os.environ.get("JWT_SECRET", "")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "")
ADMIN_USERNAME = os.environ.get("ADMIN_USERNAME", "")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD", "")
USER_DB_PATH = Path(os.environ.get("USER_DB_PATH", "/content/infosys_milestone1_users.json"))
JWT_ALGORITHM = "HS256"
JWT_TTL_MINUTES = 60
PASSWORD_HASH_ITERATIONS = 260_000
OTP_TTL_SECONDS = 300
OTP_MAX_ATTEMPTS = 5
OTP_RESEND_SECONDS = 60

# Per-process lock protects simultaneous Streamlit sessions within this server process.
DATA_LOCK = RLock()
SECURITY_QUESTIONS = [
    "What was the name of your first school?",
    "What is the name of your childhood best friend?",
    "What was the name of your first pet?",
    "In which city were you born?",
]
EMAIL_RE = re.compile(
    r"^(?=(?:[^@]*[A-Za-z]){2})[A-Za-z0-9._%+-]+@(?=(?:[^.]*[A-Za-z]){2})[A-Za-z0-9-]+(?:\.[A-Za-z]{2,})+$"
)

st.set_page_config(page_title="Infosys Milestone 1 · Authentication", page_icon="🔐", layout="centered")

# Visual-only Maritime AI premium glass UI. Authentication/application logic below is unchanged.
# The hero image is embedded so the app remains self-contained in Google Colab.
_HERO_IMAGE_DATA = "..."
st.markdown("""
<style>
:root {
  color-scheme: dark;
  --ink: #f3f8fc;
  --muted: #afc0cc;
  --glass: rgba(8, 24, 38, .64);
  --glass-border: rgba(180, 226, 240, .20);
  --aqua: #70f0e5;
  --sky: #87cfff;
  --gold: #edc58a;
}
html, body, [data-testid="stApp"], [data-testid="stAppViewContainer"] {
  background: transparent !important;
  color: var(--ink) !important;
}
.stApp {
  background: transparent !important;
  color: var(--ink) !important;
  isolation: isolate;
}
[data-testid="stAppViewContainer"] {
  min-height: 100vh;
  position: relative;
  background-image:
    linear-gradient(108deg, rgba(2, 10, 20, .93) 0%, rgba(3, 15, 26, .84) 34%, rgba(4, 15, 26, .56) 67%, rgba(2, 10, 20, .64) 100%),
    linear-gradient(0deg, rgba(2, 10, 17, .38), rgba(2, 10, 17, .10)),
    url("__HERO_IMAGE_DATA__") !important;
  background-repeat: no-repeat, no-repeat, no-repeat !important;
  background-position: center center, center center, center center !important;
  background-size: cover, cover, cover !important;
  background-attachment: fixed, fixed, fixed !important;
  overflow: hidden;
}
/* Floating glow fields remain behind controls and never intercept clicks. */
[data-testid="stAppViewContainer"]::before,
[data-testid="stAppViewContainer"]::after {
  content: "";
  position: fixed;
  display: block;
  pointer-events: none;
  z-index: 0;
  border-radius: 50%;
  filter: blur(1px);
  mix-blend-mode: screen;
  will-change: transform;
}
[data-testid="stAppViewContainer"]::before {
  width: 340px; height: 340px; left: -110px; top: 8%;
  border: 1px solid rgba(115, 239, 229, .25);
  background: radial-gradient(circle at 42% 38%, rgba(87, 229, 221, .24), rgba(67, 157, 206, .08) 50%, transparent 72%);
  box-shadow: 0 0 90px rgba(49, 207, 213, .13), inset 0 0 60px rgba(116, 242, 233, .08);
  animation: maritimeFloatA 18s ease-in-out infinite alternate;
}
[data-testid="stAppViewContainer"]::after {
  width: 270px; height: 270px; right: -70px; bottom: 7%;
  border: 1px solid rgba(132, 199, 255, .22);
  background: radial-gradient(circle at 45% 42%, rgba(96, 178, 242, .22), rgba(99, 114, 211, .07) 52%, transparent 72%);
  box-shadow: 0 0 85px rgba(76, 167, 242, .12), inset 0 0 50px rgba(137, 202, 255, .07);
  animation: maritimeFloatB 22s ease-in-out infinite alternate;
}
@keyframes maritimeFloatA {
  0% { transform: translate3d(0, 0, 0) scale(.94); }
  50% { transform: translate3d(48px, 40px, 0) scale(1.06); }
  100% { transform: translate3d(20px, -24px, 0) scale(1.01); }
}
@keyframes maritimeFloatB {
  0% { transform: translate3d(0, 0, 0) scale(1); }
  50% { transform: translate3d(-42px, -30px, 0) scale(1.08); }
  100% { transform: translate3d(-18px, 25px, 0) scale(.97); }
}
header[data-testid="stHeader"] {
  background: rgba(3, 12, 22, .40) !important;
  backdrop-filter: blur(18px);
}
section[data-testid="stMain"], [data-testid="stSidebar"] {
  position: relative;
  z-index: 1;
}
[data-testid="stSidebar"] {
  background: rgba(3, 13, 23, .80) !important;
  border-right: 1px solid rgba(170, 220, 237, .15) !important;
  backdrop-filter: blur(28px);
}
[data-testid="stSidebar"] > div { background: transparent !important; }
.block-container {
  max-width: 1080px;
  padding-top: 2.3rem;
  padding-bottom: 3.5rem;
  position: relative;
  z-index: 2;
}
h1, h2, h3, h4 {
  color: #f7fbff !important;
  font-weight: 720 !important;
  letter-spacing: -.045em;
  text-shadow: 0 2px 28px rgba(0,0,0,.30);
}
h1 { font-size: clamp(2.2rem, 4vw, 3.6rem) !important; line-height: 1.06 !important; }
h2 { font-size: clamp(1.5rem, 2.5vw, 2.15rem) !important; }
p, li, label, [data-testid="stMarkdownContainer"] { color: #e3edf4; }
[data-testid="stMarkdownContainer"] p { line-height: 1.68; }
[data-testid="stCaptionContainer"] p { color: #afc4d1 !important; }
.ui-brand {
  display: flex; align-items: center; gap: .9rem;
  margin: .15rem 0 1.35rem;
  padding: .9rem 1.05rem;
  border: 1px solid rgba(174, 224, 239, .20);
  border-radius: 18px;
  background: linear-gradient(110deg, rgba(6, 25, 39, .80), rgba(13, 39, 54, .42));
  box-shadow: 0 12px 35px rgba(0,0,0,.20), inset 0 1px 0 rgba(255,255,255,.05);
  backdrop-filter: blur(20px);
}
.ui-brand-mark {
  width: 44px; height: 44px; flex: 0 0 44px;
  display: grid; place-items: center;
  border-radius: 14px;
  border: 1px solid rgba(117, 239, 230, .40);
  background: linear-gradient(145deg, rgba(111, 239, 229, .20), rgba(80, 153, 220, .10));
  color: #a5fff5; font-size: 1.45rem;
  box-shadow: inset 0 0 18px rgba(100, 228, 224, .08), 0 0 24px rgba(72, 202, 209, .09);
}
.ui-brand-name { color: #f6fbff; font-weight: 800; letter-spacing: .16em; font-size: .95rem; }
.ui-brand-tag { color: #9fb9c9; font-size: .66rem; letter-spacing: .17em; margin-top: .22rem; }
.ui-brand-status {
  margin-left: auto; white-space: nowrap;
  border: 1px solid rgba(112, 240, 229, .25); border-radius: 999px;
  padding: .42rem .65rem; background: rgba(68, 199, 193, .08);
  color: #a5fff5; font-size: .66rem; letter-spacing: .08em; font-weight: 700;
}
.ui-eyebrow {
  display: inline-flex; align-items: center; gap: .62rem;
  padding: .5rem .82rem; margin: .1rem 0 .8rem;
  border: 1px solid rgba(137, 222, 240, .25); border-radius: 999px;
  background: rgba(12, 43, 59, .52); backdrop-filter: blur(16px);
  color: #c9f4f4; font-size: .68rem; font-weight: 750;
  letter-spacing: .15em; text-transform: uppercase;
}
.ui-eyebrow .ui-dot {
  display: inline-block; width: 7px; height: 7px; border-radius: 50%;
  background: #70f0e5; box-shadow: 0 0 13px rgba(112, 240, 229, .9);
  animation: maritimePulse 2.2s ease-in-out infinite;
}
@keyframes maritimePulse { 0%,100% { opacity: .65; transform: scale(.88); } 50% { opacity: 1; transform: scale(1.18); } }
/* Transparent glass panels */
div[data-testid="stForm"], div[data-testid="stExpander"], div[data-testid="stDataFrame"] {
  background: linear-gradient(135deg, rgba(7, 27, 42, .76), rgba(8, 25, 40, .53)) !important;
  border: 1px solid rgba(172, 224, 239, .22) !important;
  border-radius: 22px !important;
  backdrop-filter: blur(26px) saturate(135%);
  -webkit-backdrop-filter: blur(26px) saturate(135%);
  box-shadow: 0 24px 70px rgba(0,0,0,.28), inset 0 1px 0 rgba(255,255,255,.08);
}
div[data-testid="stForm"] { padding: 1.25rem 1.35rem .65rem; transition: border-color .25s ease, box-shadow .25s ease, transform .25s ease; }
div[data-testid="stForm"]:hover {
  border-color: rgba(119, 237, 231, .35) !important;
  box-shadow: 0 28px 80px rgba(0,0,0,.34), 0 0 32px rgba(59,195,206,.06), inset 0 1px 0 rgba(255,255,255,.10);
}
/* High-contrast inputs: force readable text even when browser autofill is active. */
div[data-testid="stTextInput"] input,
div[data-testid="stTextArea"] textarea,
div[data-testid="stNumberInput"] input,
div[data-baseweb="input"] input,
input[type="text"], input[type="password"], input[type="email"], textarea {
  color: #f7fbff !important;
  -webkit-text-fill-color: #f7fbff !important;
  caret-color: #79f5ea !important;
  background: rgba(3, 17, 30, .94) !important;
  border: 1px solid rgba(168, 220, 238, .27) !important;
  border-radius: 12px !important;
  min-height: 2.85rem;
  transition: border-color .2s ease, box-shadow .2s ease, background .2s ease;
}
div[data-testid="stTextInput"] [data-baseweb="input"],
div[data-testid="stTextArea"] [data-baseweb="textarea"],
div[data-baseweb="input"] {
  background: rgba(3, 17, 30, .94) !important;
  border-radius: 12px !important;
  border-color: rgba(168, 220, 238, .27) !important;
}
div[data-testid="stTextInput"] input:focus,
div[data-testid="stTextArea"] textarea:focus,
div[data-testid="stNumberInput"] input:focus,
div[data-baseweb="input"] input:focus,
input[type="text"]:focus, input[type="password"]:focus, input[type="email"]:focus {
  border-color: #72e6e2 !important;
  box-shadow: 0 0 0 .15rem rgba(83, 220, 220, .15) !important;
  background: rgba(4, 22, 36, .98) !important;
}
input::placeholder, textarea::placeholder { color: #91a8b8 !important; -webkit-text-fill-color: #91a8b8 !important; opacity: 1 !important; }
input:-webkit-autofill, input:-webkit-autofill:hover, input:-webkit-autofill:focus {
  -webkit-text-fill-color: #f7fbff !important;
  -webkit-box-shadow: 0 0 0 1000px #061623 inset !important;
  transition: background-color 9999s ease-out 0s;
}
/* Selectbox and its expanded options */
[data-testid="stSelectbox"] [role="combobox"], [data-baseweb="select"] > div {
  color: #f4fbff !important; background: rgba(3, 17, 30, .92) !important;
  border-color: rgba(168, 220, 238, .25) !important; border-radius: 12px !important;
}
[role="listbox"], [role="option"] { background: #0a1c2a !important; color: #f2f8fc !important; }
[role="option"]:hover { background: #15394a !important; }
/* Aqua-to-ice action buttons */
.stButton button, [data-testid="stFormSubmitButton"] button, [data-testid="stDownloadButton"] button {
  border: 1px solid rgba(145, 247, 241, .60) !important;
  border-radius: 12px !important;
  background: linear-gradient(110deg, #9cf5ec, #b3e8ff) !important;
  color: #03131d !important; font-weight: 750 !important; min-height: 2.85rem;
  transition: transform .22s ease, box-shadow .22s ease, filter .22s ease;
  box-shadow: 0 8px 26px rgba(27, 176, 188, .15);
}
.stButton button p, [data-testid="stFormSubmitButton"] button p, [data-testid="stDownloadButton"] button p { color: #03131d !important; }
.stButton button:hover, [data-testid="stFormSubmitButton"] button:hover, [data-testid="stDownloadButton"] button:hover {
  transform: translateY(-2px); filter: brightness(1.05);
  box-shadow: 0 13px 34px rgba(27, 202, 202, .22); border-color: #d2fffb !important;
}
.stButton button:active, [data-testid="stFormSubmitButton"] button:active { transform: translateY(0); }
[data-testid="stAlert"] {
  border-radius: 15px !important; border: 1px solid rgba(168, 220, 238, .20) !important;
  background: rgba(6, 28, 43, .82) !important; backdrop-filter: blur(18px); color: #f3f9fc !important;
}
[data-testid="stAlert"] p, [data-testid="stAlert"] span { color: #f3f9fc !important; }
.stTabs [data-baseweb="tab-list"] { gap: .35rem; background: rgba(4, 21, 34, .65); padding: .35rem; border-radius: 14px; border: 1px solid rgba(168,220,238,.13); }
.stTabs [data-baseweb="tab"] { border-radius: 10px; color: #acc0cd !important; }
.stTabs [aria-selected="true"] { background: rgba(105, 230, 224, .16) !important; color: #f5ffff !important; }
.stTabs [data-baseweb="tab-highlight"] { background: #7be7e0 !important; }
[data-testid="stTable"] table { border-color: rgba(168,220,238,.15) !important; }
[data-testid="stDataFrame"] * { color: #eef8fc; }
hr { border-color: rgba(168,220,238,.15) !important; }
a { color: #9ff8ef !important; text-decoration-color: rgba(159,248,239,.52) !important; }
a:hover { text-decoration-color: #ffffff !important; }
@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { animation-duration: .01ms !important; animation-iteration-count: 1 !important; transition-duration: .01ms !important; scroll-behavior: auto !important; }
}
@media (max-width: 640px) {
  .block-container { padding-top: 1.4rem; padding-left: 1rem; padding-right: 1rem; }
  div[data-testid="stForm"] { padding: 1rem .85rem .45rem; border-radius: 18px !important; }
  .ui-brand { gap: .65rem; padding: .75rem; }
  .ui-brand-status { display: none; }
  [data-testid="stAppViewContainer"]::before { width: 240px; height: 240px; left: -120px; }
  [data-testid="stAppViewContainer"]::after { width: 210px; height: 210px; right: -110px; }
}
</style>
""".replace("__HERO_IMAGE_DATA__", _HERO_IMAGE_DATA), unsafe_allow_html=True)


if len(JWT_SECRET) < 32:
    st.error("Configuration missing or incomplete: add a JWT_SECRET of at least 32 characters in Colab Secrets and enable notebook access.")
    st.stop()
if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
    st.error("Email configuration is missing. Add EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets and enable notebook access.")
    st.stop()
if not ADMIN_USERNAME or not ADMIN_PASSWORD:
    st.error("Admin configuration is missing. Add ADMIN_USERNAME and ADMIN_PASSWORD in Colab Secrets and enable notebook access.")
    st.stop()


def read_users_unlocked():
    if not USER_DB_PATH.exists():
        return {}
    try:
        with USER_DB_PATH.open("r", encoding="utf-8") as handle:
            data = json.load(handle)
        return data if isinstance(data, dict) else {}
    except (OSError, json.JSONDecodeError):
        return {}


def write_users_unlocked(users):
    USER_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    temp_path = USER_DB_PATH.with_suffix(USER_DB_PATH.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        json.dump(users, handle, ensure_ascii=False, indent=2)
    os.replace(temp_path, USER_DB_PATH)


def get_users():
    with DATA_LOCK:
        return read_users_unlocked()


def normalize_username(username):
    return username.strip().casefold()


def normalize_email(email):
    return email.strip().casefold()


def hash_value(value):
    salt = secrets.token_bytes(16)
    digest = hashlib.pbkdf2_hmac("sha256", value.encode("utf-8"), salt, PASSWORD_HASH_ITERATIONS)
    return f"pbkdf2_sha256${PASSWORD_HASH_ITERATIONS}${salt.hex()}${digest.hex()}"


def verify_value(value, stored_hash):
    try:
        algorithm, iteration_text, salt_hex, expected_hex = stored_hash.split("$", 3)
        if algorithm != "pbkdf2_sha256":
            return False
        iterations = int(iteration_text)
        if iterations < 100_000 or iterations > 2_000_000:
            return False
        salt = bytes.fromhex(salt_hex)
        expected = bytes.fromhex(expected_hex)
        actual = hashlib.pbkdf2_hmac("sha256", value.encode("utf-8"), salt, iterations)
        return hmac.compare_digest(actual, expected)
    except (ValueError, TypeError):
        return False


def password_errors(password):
    errors = []
    if len(password) < 8:
        errors.append("Use at least 8 characters.")
    if not any(char.isupper() for char in password):
        errors.append("Include at least one uppercase letter.")
    if not any(char.islower() for char in password):
        errors.append("Include at least one lowercase letter.")
    if not any(char.isdigit() for char in password):
        errors.append("Include at least one number.")
    if not any(not char.isalnum() and not char.isspace() for char in password):
        errors.append("Include at least one special symbol (not a space).")
    return errors


def find_user_record(identifier, users=None):
    users = users if users is not None else get_users()
    candidate = identifier.strip()
    if not candidate:
        return None, None
    key = normalize_username(candidate)
    if key in users:
        return key, users[key]
    candidate_email = normalize_email(candidate)
    for stored_key, record in users.items():
        if normalize_email(record.get("email", "")) == candidate_email:
            return stored_key, record
    return None, None


def issue_jwt(username):
    now = datetime.now(timezone.utc)
    payload = {
        "sub": username,
        "iat": now,
        "exp": now + timedelta(minutes=JWT_TTL_MINUTES),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)


def authenticated_username():
    token = st.session_state.get("jwt_token")
    if not token:
        return None
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
        username = payload.get("sub")
        users = get_users()
        if not username or normalize_username(username) not in users:
            clear_user_session()
            return None
        return users[normalize_username(username)]["username"]
    except jwt.PyJWTError:
        clear_user_session()
        return None


def clear_user_session():
    st.session_state.pop("jwt_token", None)
    st.session_state.pop("jwt_username", None)


def set_page(name):
    st.session_state["page"] = name


def send_otp_email(recipient, otp):
    message = EmailMessage()
    message["Subject"] = "Your password reset verification code"
    message["From"] = EMAIL_ADDRESS
    message["To"] = recipient
    message.set_content(
        f"Your password reset code is: {otp}\n\n"
        f"It expires in 5 minutes and can be used only once. If you did not request this, ignore this email."
    )
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as smtp:
        smtp.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        smtp.send_message(message)


def clear_otp_state():
    for key in (
        "otp_email", "otp_hash", "otp_expires_at", "otp_attempts",
        "otp_last_sent_at", "otp_verified_email", "otp_verified_until",
    ):
        st.session_state.pop(key, None)


def reset_password_for_key(user_key, new_password):
    with DATA_LOCK:
        users = read_users_unlocked()
        if user_key not in users:
            return False
        users[user_key]["password_hash"] = hash_value(new_password)
        write_users_unlocked(users)
    return True


# Initialize navigation state before displaying the sidebar.
st.session_state.setdefault("page", "Login")
username_logged_in = authenticated_username()
if username_logged_in:
    st.session_state["jwt_username"] = username_logged_in
    allowed_pages = ["Dashboard", "Logout"]
    if st.session_state.get("page") not in allowed_pages:
        st.session_state["page"] = "Dashboard"
else:
    # Do not allow navigation directly to a protected dashboard without a valid JWT.
    if st.session_state.get("page") == "Dashboard":
        st.session_state["page"] = "Login"

with st.sidebar:
    st.title("🔐 Milestone 1")
    st.caption("User Authentication Module")
    if username_logged_in:
        st.write(f"Signed in as **{username_logged_in}**")
        if st.button("Dashboard", use_container_width=True, key="nav_dashboard"):
            set_page("Dashboard")
            st.rerun()
        if st.button("Logout", use_container_width=True, key="nav_logout"):
            clear_user_session()
            st.session_state.pop("admin_authenticated", None)
            st.session_state["page"] = "Login"
            st.rerun()
    else:
        for label in ("Login", "Sign Up", "Forgot Password", "Admin Login"):
            if st.button(label, use_container_width=True, key="nav_" + label.lower().replace(" ", "_")):
                set_page(label)
                st.rerun()
    st.divider()
    st.caption("Demo project. Use test accounts only; do not enter sensitive personal information.")

# Show one-time flash messages after navigation.
flash = st.session_state.pop("flash_message", None)
if flash:
    st.success(flash)

page = st.session_state.get("page", "Login")

if page == "Login":
    st.markdown("""
    <div class="ui-brand">
      <div class="ui-brand-mark">⌁</div>
      <div>
        <div class="ui-brand-name">MARITIME AI</div>
        <div class="ui-brand-tag">INTELLIGENT OCEAN OPERATIONS</div>
      </div>
      <div class="ui-brand-status">● SECURE ACCESS</div>
    </div>
    <div class="ui-eyebrow"><span class="ui-dot"></span> Identity gateway &nbsp;·&nbsp; Protected session</div>
    """, unsafe_allow_html=True)
    st.title("Login")
    st.write("Sign in using your username or registered email address.")
    with st.form("login_form"):
        identifier = st.text_input("Username or Email")
        password = st.text_input("Password", type="password")
        submitted = st.form_submit_button("Login", use_container_width=True)
    if submitted:
        if not identifier.strip() or not password:
            st.warning("Please complete all required fields.")
        else:
            users = get_users()
            user_key, record = find_user_record(identifier, users)
            # Keep the public error generic regardless of which credential failed.
            if record and verify_value(password, record.get("password_hash", "")):
                st.session_state["jwt_token"] = issue_jwt(record["username"])
                st.session_state["jwt_username"] = record["username"]
                st.session_state["page"] = "Dashboard"
                st.rerun()
            else:
                st.error("Invalid username/email or password.")

elif page == "Sign Up":
    st.title("Create an account")
    st.write("All fields are required. Passwords are stored as salted hashes.")
    with st.form("signup_form"):
        username = st.text_input("Username")
        email = st.text_input("Email address")
        password = st.text_input("Password", type="password")
        confirm_password = st.text_input("Confirm password", type="password")
        security_question = st.selectbox("Security question", SECURITY_QUESTIONS)
        security_answer = st.text_input("Security answer", type="password")
        submitted = st.form_submit_button("Create account", use_container_width=True)
    if submitted:
        if not all([username.strip(), email.strip(), password, confirm_password, security_question, security_answer.strip()]):
            st.warning("Please complete all required fields.")
        elif len(username.strip()) < 2:
            st.error("Username must contain at least 2 characters.")
        elif not EMAIL_RE.fullmatch(email.strip()):
            st.error("Enter an email with at least 2 letters before @, at least 2 letters in the domain, and at least 2 letters after the final dot (for example, ab@cd.ef).")
        elif password != confirm_password:
            st.error("Password and Confirm Password must match exactly.")
        elif password_errors(password):
            st.error("Password does not meet the required rules:\n\n- " + "\n- ".join(password_errors(password)))
        else:
            user_key = normalize_username(username)
            clean_email = email.strip()
            with DATA_LOCK:
                users = read_users_unlocked()
                username_exists = user_key in users
                email_exists = any(normalize_email(item.get("email", "")) == normalize_email(clean_email) for item in users.values())
                if username_exists:
                    error_message = "That username is already registered. Choose another username."
                elif email_exists:
                    error_message = "That email is already registered. Try logging in or use Forgot Password."
                else:
                    users[user_key] = {
                        "username": username.strip(),
                        "email": clean_email,
                        "password_hash": hash_value(password),
                        "security_question": security_question,
                        "security_answer_hash": hash_value(security_answer.strip().casefold()),
                        "created_at": datetime.now(timezone.utc).isoformat(),
                    }
                    write_users_unlocked(users)
                    error_message = None
            if error_message:
                st.error(error_message)
            else:
                st.session_state["flash_message"] = "Account created successfully. Please log in."
                st.session_state["page"] = "Login"
                st.rerun()

elif page == "Forgot Password":
    st.title("Forgot password")
    st.write("Choose either the security-question route or email OTP route.")
    security_tab, otp_tab = st.tabs(["Security question", "Email OTP"])

    with security_tab:
        st.subheader("Reset with your security question")
        with st.form("lookup_security_form"):
            lookup_username = st.text_input("Username", key="security_lookup_username")
            lookup_submitted = st.form_submit_button("Find security question", use_container_width=True)
        if lookup_submitted:
            if not lookup_username.strip():
                st.warning("Enter your username.")
            else:
                found_key, found_record = find_user_record(lookup_username)
                if found_record:
                    st.session_state["security_recovery_key"] = found_key
                    st.session_state["security_recovery_question"] = found_record["security_question"]
                    st.rerun()
                else:
                    st.error("We could not verify that account. Check the username and try again.")

        recovery_key = st.session_state.get("security_recovery_key")
        if recovery_key:
            users_now = get_users()
            record = users_now.get(recovery_key)
            if record:
                st.info("Security question: " + record["security_question"])
                with st.form("security_reset_form"):
                    answer = st.text_input("Security answer", type="password")
                    new_password = st.text_input("New password", type="password")
                    confirm_new_password = st.text_input("Confirm new password", type="password")
                    reset_submitted = st.form_submit_button("Reset password", use_container_width=True)
                if reset_submitted:
                    if not answer.strip() or not new_password or not confirm_new_password:
                        st.warning("Please complete all required fields.")
                    elif new_password != confirm_new_password:
                        st.error("New password and confirmation must match exactly.")
                    elif password_errors(new_password):
                        st.error("Password does not meet the required rules:\n\n- " + "\n- ".join(password_errors(new_password)))
                    elif not verify_value(answer.strip().casefold(), record.get("security_answer_hash", "")):
                        st.error("The answer could not be verified. Try again.")
                    elif reset_password_for_key(recovery_key, new_password):
                        st.session_state.pop("security_recovery_key", None)
                        st.session_state.pop("security_recovery_question", None)
                        clear_otp_state()
                        st.session_state["flash_message"] = "Password reset successfully. Please log in with your new password."
                        st.session_state["page"] = "Login"
                        st.rerun()
                    else:
                        st.error("Could not reset the password. Please try again.")

    with otp_tab:
        st.subheader("Reset with an email OTP")
        st.caption("The code expires after 5 minutes and is limited to 5 verification attempts. Resending is limited to one request per minute in this session.")
        with st.form("send_otp_form"):
            otp_email_input = st.text_input("Registered email address", key="otp_email_input")
            send_submitted = st.form_submit_button("Send OTP", use_container_width=True)
        if send_submitted:
            candidate = otp_email_input.strip()
            now = time.time()
            if not candidate:
                st.warning("Enter your registered email address.")
            elif not EMAIL_RE.fullmatch(candidate):
                st.error("Enter a valid email address.")
            elif now - st.session_state.get("otp_last_sent_at", 0) < OTP_RESEND_SECONDS:
                st.warning("Please wait before requesting another OTP.")
            else:
                users = get_users()
                email_key, email_record = find_user_record(candidate, users)
                if not email_record or normalize_email(email_record.get("email", "")) != normalize_email(candidate):
                    # Generic response avoids confirming whether an email is registered.
                    st.info("If an account exists for this email, an OTP will be sent.")
                    st.session_state["otp_last_sent_at"] = now
                else:
                    otp = f"{secrets.randbelow(1_000_000):06d}"
                    try:
                        send_otp_email(email_record["email"], otp)
                        st.session_state["otp_email"] = normalize_email(email_record["email"])
                        st.session_state["otp_hash"] = hashlib.sha256(otp.encode("ascii")).hexdigest()
                        st.session_state["otp_expires_at"] = now + OTP_TTL_SECONDS
                        st.session_state["otp_attempts"] = 0
                        st.session_state.pop("otp_verified_email", None)
                        st.session_state.pop("otp_verified_until", None)
                        st.session_state["otp_last_sent_at"] = now
                        st.success("If an account exists for this email, an OTP has been sent. Check your inbox and spam folder.")
                    except (smtplib.SMTPException, OSError) as exc:
                        # Do not display SMTP credentials or sensitive exception details.
                        st.error("We could not send the email. Check the Gmail App Password and try again.")

        otp_email = st.session_state.get("otp_email")
        otp_hash = st.session_state.get("otp_hash")
        otp_expires_at = st.session_state.get("otp_expires_at", 0)
        if otp_email and otp_hash and time.time() < otp_expires_at:
            with st.form("verify_otp_form"):
                otp_entered = st.text_input("6-digit OTP", max_chars=6)
                verify_submitted = st.form_submit_button("Verify OTP", use_container_width=True)
            if verify_submitted:
                if st.session_state.get("otp_attempts", 0) >= OTP_MAX_ATTEMPTS:
                    clear_otp_state()
                    st.error("Too many attempts. Request a new OTP.")
                elif not otp_entered or not otp_entered.isdigit() or len(otp_entered) != 6:
                    st.warning("Enter the 6-digit code from your email.")
                elif hmac.compare_digest(hashlib.sha256(otp_entered.encode("ascii")).hexdigest(), otp_hash):
                    st.session_state["otp_verified_email"] = otp_email
                    st.session_state["otp_verified_until"] = time.time() + OTP_TTL_SECONDS
                    st.session_state.pop("otp_hash", None)  # one-time: code cannot be verified again
                    st.session_state["otp_expires_at"] = 0
                    st.success("Email verified. Set your new password below.")
                    st.rerun()
                else:
                    st.session_state["otp_attempts"] = st.session_state.get("otp_attempts", 0) + 1
                    if st.session_state["otp_attempts"] >= OTP_MAX_ATTEMPTS:
                        clear_otp_state()
                        st.error("Too many incorrect attempts. Request a new OTP.")
                    else:
                        st.error(f"Incorrect OTP. Attempts remaining: {OTP_MAX_ATTEMPTS - st.session_state['otp_attempts']}.")
        elif otp_email and otp_hash and time.time() >= otp_expires_at:
            st.session_state.pop("otp_hash", None)
            st.error("Your OTP has expired. Request a new one.")

        verified_email = st.session_state.get("otp_verified_email")
        verified_until = st.session_state.get("otp_verified_until", 0)
        if verified_email and time.time() < verified_until:
            with st.form("otp_password_reset_form"):
                new_password_otp = st.text_input("New password", type="password")
                confirm_password_otp = st.text_input("Confirm new password", type="password")
                reset_otp_submitted = st.form_submit_button("Set new password", use_container_width=True)
            if reset_otp_submitted:
                if not new_password_otp or not confirm_password_otp:
                    st.warning("Please complete all required fields.")
                elif new_password_otp != confirm_password_otp:
                    st.error("New password and confirmation must match exactly.")
                elif password_errors(new_password_otp):
                    st.error("Password does not meet the required rules:\n\n- " + "\n- ".join(password_errors(new_password_otp)))
                else:
                    users = get_users()
                    email_key = None
                    for key, record in users.items():
                        if normalize_email(record.get("email", "")) == verified_email:
                            email_key = key
                            break
                    if email_key and time.time() < st.session_state.get("otp_verified_until", 0) and reset_password_for_key(email_key, new_password_otp):
                        clear_otp_state()
                        st.session_state["flash_message"] = "Password reset successfully. Please log in with your new password."
                        st.session_state["page"] = "Login"
                        st.rerun()
                    else:
                        clear_otp_state()
                        st.error("Verification expired or account unavailable. Start the recovery process again.")
        elif verified_email:
            clear_otp_state()
            st.warning("Email verification expired. Request a new OTP.")

elif page == "Dashboard":
    if not username_logged_in:
        st.session_state["page"] = "Login"
        st.warning("Please log in to access the dashboard.")
        st.stop()
    users = get_users()
    key = normalize_username(username_logged_in)
    record = users.get(key)
    st.title("User Dashboard")
    st.success(f"Welcome, {username_logged_in}!")
    if record:
        st.write(f"**Registered email:** {record.get('email', '')}")
    st.info("You are viewing the protected dashboard. Your JWT is checked before this page is shown.")
    if st.button("Logout", type="primary"):
        clear_user_session()
        st.session_state["page"] = "Login"
        st.rerun()

elif page == "Admin Login":
    st.title("Admin Login")
    st.write("This is a separate admin account; regular signups cannot access this page.")
    with st.form("admin_login_form"):
        admin_username = st.text_input("Admin username")
        admin_password = st.text_input("Admin password", type="password")
        admin_submitted = st.form_submit_button("Admin sign in", use_container_width=True)
    if admin_submitted:
        user_match = hmac.compare_digest(admin_username, ADMIN_USERNAME)
        password_match = hmac.compare_digest(admin_password, ADMIN_PASSWORD)
        if user_match and password_match:
            st.session_state["admin_authenticated"] = True
            st.session_state["page"] = "Admin Dashboard"
            st.rerun()
        else:
            st.error("Invalid admin username or password.")

elif page == "Admin Dashboard":
    if not st.session_state.get("admin_authenticated", False):
        st.session_state["page"] = "Admin Login"
        st.warning("Admin login is required.")
        st.stop()
    st.title("Admin Dashboard")
    st.write("Registered users (passwords and password hashes are never displayed).")
    users = get_users()
    records = [{"Username": item.get("username", ""), "Email": item.get("email", "")} for item in users.values()]
    if records:
        st.dataframe(records, use_container_width=True, hide_index=True)
    else:
        st.info("No registered users yet.")
    if st.button("Admin logout", type="primary"):
        st.session_state.pop("admin_authenticated", None)
        st.session_state["page"] = "Admin Login"
        st.rerun()

else:
    st.session_state["page"] = "Login"
    st.rerun()


Writing /content/app.py


## Read secrets and launch Streamlit + ngrok

Run this cell after you have added all six secrets and enabled notebook access for them. It does not print secret values. It prints only the public ngrok URL needed to open the app. Do not include cell output in your submitted notebook.

In [ ]:
import os, sys, time, socket, subprocess
from google.colab import userdata
from pyngrok import ngrok

REQUIRED_SECRETS = [
    "JWT_SECRET",
    "NGROK_AUTHTOKEN",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD",
    "ADMIN_USERNAME",
    "ADMIN_PASSWORD",
]
secret_values = {}
missing_secrets = []
for secret_name in REQUIRED_SECRETS:
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None
    if not secret_value or not str(secret_value).strip():
        missing_secrets.append(secret_name)
    else:
        secret_values[secret_name] = str(secret_value).strip()

if missing_secrets:
    raise RuntimeError(
        "Missing or inaccessible Colab Secrets: " + ", ".join(missing_secrets) +
        ". Add them using the Secrets (key) panel and enable notebook access."
    )
if len(secret_values["JWT_SECRET"]) < 32:
    raise ValueError("JWT_SECRET must be at least 32 characters long. Replace it in Colab Secrets.")

# Google may display app passwords grouped by spaces; remove those spaces before SMTP login.
secret_values["EMAIL_PASSWORD"] = secret_values["EMAIL_PASSWORD"].replace(" ", "")

# Disconnect any existing tunnels first to prevent "endpoint already online" errors
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
except Exception:
    pass

# Stop an app started by a previous run of this cell, if any.
try:
    if "streamlit_process" in globals() and streamlit_process.poll() is None:
        streamlit_process.terminate()
        try:
            streamlit_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            streamlit_process.kill()
except Exception:
    pass
try:
    if "streamlit_log_handle" in globals() and not streamlit_log_handle.closed:
        streamlit_log_handle.close()
except Exception:
    pass
try:
    ngrok.kill()
except Exception:
    pass

app_env = os.environ.copy()
for secret_name, secret_value in secret_values.items():
    app_env[secret_name] = secret_value
app_env["USER_DB_PATH"] = "/content/infosys_milestone1_users.json"
app_env["STREAMLIT_SERVER_HEADLESS"] = "true"
app_env["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"

streamlit_log_handle = open("/content/streamlit.log", "w", encoding="utf-8")
streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "/content/app.py",
     "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true"],
    env=app_env,
    stdout=streamlit_log_handle,
    stderr=subprocess.STDOUT,
)

# Wait for Streamlit to listen on port 8501 without printing log content or secrets.
server_ready = False
for _ in range(30):
    if streamlit_process.poll() is not None:
        break
    try:
        with socket.create_connection(("127.0.0.1", 8501), timeout=1):
            server_ready = True
            break
    except OSError:
        time.sleep(1)

if not server_ready:
    try:
        streamlit_process.terminate()
    except Exception:
        pass
    raise RuntimeError("Streamlit did not start. Inspect /content/streamlit.log in Colab for the error; do not share secrets from the runtime.")

try:
    ngrok.set_auth_token(secret_values["NGROK_AUTHTOKEN"])
except Exception:
    # Pass silently if the authtoken is already configured in ngrok.yml
    pass

tunnel = ngrok.connect(addr=8501, proto="http")
print("Open your app at:", tunnel.public_url)
print("Keep this Colab runtime running while testing. Stop the tunnel and app after testing.")

KeyboardInterrupt: 

## Stop the app when you finish testing

Run this cleanup cell after capturing your screenshots. It stops the Streamlit process and ngrok tunnel. Before submitting, use **Edit → Clear all outputs**, save/download the `.ipynb`, then verify that no secrets, OTP codes or public URLs remain in the notebook.

In [ ]:
try:
    ngrok.kill()
except Exception:
    pass
try:
    if "streamlit_process" in globals() and streamlit_process.poll() is None:
        streamlit_process.terminate()
        try:
            streamlit_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            streamlit_process.kill()
except Exception:
    pass
try:
    if "streamlit_log_handle" in globals() and not streamlit_log_handle.closed:
        streamlit_log_handle.close()
except Exception:
    pass
print("Streamlit process and ngrok tunnel have been stopped if they were running.")


Streamlit process and ngrok tunnel have been stopped if they were running.
